# 03 – Inference
Load the packaged pipeline and score new patients – exactly what the API does.

In [1]:
import sys, pathlib

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]
import os

os.chdir(ROOT)

In [2]:
import json
from heart_risk.predict import load_model, load_meta, predict_records

model = load_model()
load_meta()

{'model_name': 'logistic_regression',
 'model_version': '1.0.0',
 'trained_at': '2026-10-16T20:59:10.013029+00:00',
 'selection_metric': 'cv_roc_auc_mean',
 'best_params': {'model__C': 1.0,
  'model__class_weight': 'balanced',
  'model__penalty': 'l2'},
 'cv_metrics': {'cv_accuracy_mean': 0.8552,
  'cv_precision_mean': 0.8608,
  'cv_recall_mean': 0.8281,
  'cv_f1_mean': 0.8404,
  'cv_roc_auc_mean': 0.9064,
  'cv_accuracy_std': 0.0301,
  'cv_precision_std': 0.0754,
  'cv_recall_std': 0.0542,
  'cv_f1_std': 0.0296,
  'cv_roc_auc_std': 0.0197},
 'holdout_metrics': {'test_accuracy': 0.8852,
  'test_precision': 0.8182,
  'test_recall': 0.9643,
  'test_f1': 0.8852,
  'test_roc_auc': 0.9567},
 'mlflow_run_id': '6b1426b88735410f8aba8e65d888c574',
 'n_training_rows': 303,
 'features': ['age',
  'trestbps',
  'chol',
  'thalach',
  'oldpeak',
  'ca',
  'sex',
  'fbs',
  'exang',
  'cp',
  'restecg',
  'slope',
  'thal'],
 'quick': False}

In [3]:
patients = json.load(open("samples/batch.json"))["patients"]
preds = predict_records(model, patients)
preds

[{'prediction': 1,
  'label': 'heart disease',
  'probability_disease': 0.9972,
  'confidence': 0.9972,
  'risk_band': 'high'},
 {'prediction': 0,
  'label': 'no heart disease',
  'probability_disease': 0.0523,
  'confidence': 0.9477,
  'risk_band': 'low'},
 {'prediction': 0,
  'label': 'no heart disease',
  'probability_disease': 0.0137,
  'confidence': 0.9863,
  'risk_band': 'low'}]

The third patient omits `ca` and `thal`; the pipeline's imputers fill them in, so partial records are still scored.

In [4]:
# Same artefact in MLflow pyfunc format
import mlflow.pyfunc, pandas as pd
from heart_risk import config

pyf = mlflow.pyfunc.load_model(str(config.MLFLOW_MODEL_DIR))
pyf.predict(pd.DataFrame(patients, columns=config.INPUT_FEATURES).astype(float))  # signature expects doubles

array([1, 0, 0])